# Exercise 1.3.2.9 — generalize results to another task (optional)

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `1.3.2 Function Vectors & Model Steering`  
**Notebook:** `1.3.2_Function_Vectors_&_Model_Steering_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=1.3.2.9](https://delta-drills.vercel.app/?arena_exercise=1.3.2.9)


# [1.3.2] Function Vectors & Model Steering (exercises)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/12_[1.3.2]_Function_Vectors_&_Model_Steering)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part32_function_vectors_and_model_steering/1.3.2_Function_Vectors_&_Model_Steering_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part32_function_vectors_and_model_steering/1.3.2_Function_Vectors_&_Model_Steering_solutions.ipynb?t=20260329)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

> *Note - if you get a numpy-related error at any point (e.g. when running the import cell for the first time, or running the first numpy function), you should restart the kernel and run the setup code again. The error should go away.*

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-14-2.png" width="350">

# Introduction

These exercises serve as an exploration of the following question: ***can we steer a model to produce different outputs / have a different behaviour, by intervening on the model's forward pass using vectors found by non gradient descent-based methods?***

The majority of the exercises focus on [function vectors](https://functions.baulab.info/): vectors which are extracted from forward passes on in-context learning (ICL) tasks, and added to the residual stream in order to trigger the execution of this task from a zero-shot prompt. The diagram below illustrates this.

<img src="https://functions.baulab.info/images/Paper/fv-demonstrations.png" width="650">

The exercises also take you through use of the `nnsight` library, which is designed to support this kind of work (and other interpretability research) on very large language models - i.e. larger than models like GPT2-Small which you might be used to at this point in the course.

The final set of exercises look at Alex Turner et al's work on [steering vectors](https://www.lesswrong.com/posts/5spBue2z2tw4JuDCx/steering-gpt-2-xl-by-adding-an-activation-vector), which is conceptually related but has different aims and methodologies.

## Content & Learning Objectives

### 1️⃣ Introduction to `nnsight`

In this section, you'll learn the basics of how to use the `nnsight` library: running forward passes on your model, and saving the internal states. You'll also learn some basics of HuggingFace models which translate over into `nnsight` models (e.g. tokenization, and how to work with model output).

> ##### Learning Objectives
>
> * Learn the basics of the `nnsight` library, and what it can be useful for
> * Learn some basics of HuggingFace models (e.g. tokenization, model output)
> * Use it to extract & visualise GPT-J-6B's internal activations

### 2️⃣ Task-encoding hidden states

We'll begin with the following question, posed by the Function Vectors paper:

> *When a transformer processes an ICL (in-context-learning) prompt with exemplars demonstrating task $T$, do any hidden states encode the task itself?*

We'll prove that the answer is yes, by constructing a vector $h$ from a set of ICL prompts for the **antonym task**, and intervening with our vector to make our model produce antonyms on zero-shot prompts.

This will require you to learn how to perform causal interventions with `nnsight`, not just save activations.

(Note - this section structurally follows section 2.1 of the function vectors paper).

> ##### Learning Objectives
>
> * Understand how `nnsight` can be used to perform causal interventions, and perform some yourself
> * Reproduce the "h-vector results" from the function vectors paper; that the residual stream does contain a vector which encodes the task and can induce task behaviour on zero-shot prompts


### 3️⃣ Function Vectors

In this section, we'll replicate the crux of the paper's results, by identifying a set of attention heads whose outputs have a large effect on the model's ICL performance, and showing we can patch with these vectors to induce task-solving behaviour on randomly shuffled prompts.

We'll also learn how to use `nnsight` for multi-token generation, and steer the model's behaviour. There exist exercises where you can try this out for different tasks, e.g. the Country-Capitals task, where you'll be able to steer the model to complete prompts like `"When you think of Netherlands, you usually think of"` by talking about Amsterdam.

(Note - this section structurally follows sections 2.2, 2.3 and some of section 3 from the function vectors paper).

> ##### Learning Objectives
>
> * Define a metric to measure the causal effect of each attention head on the correct performance of the in-context learning task
> * Understand how to rearrange activations in a model during an `nnsight` forward pass, to extract activations corresponding to a particular attention head
> * Learn how to use `nnsight` for multi-token generation

### 4️⃣ Steering Vectors in GPT2-XL

Here, we discuss a different but related set of research: Alex Turner's work on steering vectors. This also falls under the umbrella of "interventions in the residual stream using vectors found with forward pass (non-SGD) based methods in order to alter behaviour", but it has a different setup, objectives, and approach.

> ##### Learning Objectives
>
> * Understand the goals & main results from Alex Turner et al's work on steering vectors
> * Reproduce the changes in behaviour described in their initial post

### ☆ Bonus

Lastly, we discuss some possible extensions of function vectors & steering vectors work, which is currently an exciting area of development (e.g. with a paper on steering Llama 2-13b coming out as recently as December 2023).

## Setup code

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import nnsight
except:
    %pip install openai>=1.56.2 nnsight einops jaxtyping plotly transformer_lens==2.17.0 git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python gradio typing-extensions
    %pip install --upgrade pydantic

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import logging
import os
import sys
import time
from collections import defaultdict
from pathlib import Path

import circuitsvis as cv
import einops
import numpy as np
import torch as t
from IPython.display import display
from jaxtyping import Float
from nnsight import CONFIG, LanguageModel
from openai import OpenAI
from rich import print as rprint
from rich.table import Table
from torch import Tensor

# Hide some info logging messages from nnsight
logging.disable(sys.maxsize)

t.set_grad_enabled(False)
device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part32_function_vectors_and_model_steering"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part32_function_vectors_and_model_steering.solutions as solutions
import part32_function_vectors_and_model_steering.tests as tests
from plotly_utils import imshow

MAIN = __name__ == "__main__"

# 1️⃣ Introduction to `nnsight`

> ##### Learning Objectives
>
> * Learn the basics of the `nnsight` library, and what it can be useful for
> * Learn some basics of HuggingFace models (e.g. tokenization, model output)
> * Use it to extract & visualise GPT-J-6B's internal activations

## Remote execution

We'll start by discussing [remote execution]((https://nnsight.net/notebooks/features/remote_execution/)) - the ability `nnsight` has to run models on an external server, which is one of the major benefits of the library as a research tool. This helps you bypass the memory & computational limits you might be faced with on your own machine. For remote execution to work, you need 2 things:

1. An API key fromm the community Discord, which you can request [here](https://login.ndif.us/) (use Google as an identity provider, if no other provider is more appropriate for you)
2. The model you're working with being live - you can see all live models in the status page [here](https://nnsight.net/status/)

Note that the status page sometimes takes ~5 minutes to load all live models - click the dropdown below to see an example of what the status page should look like once the models have loaded. If you can't see the model you're looking for in this list, then you should set `REMOTE=False` for these exercises, or else make a request to the NDIF Discord to get the model live.

<details>
<summary>Example status page</summary>

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/ndif-status.png" width="650">

</details>

## Important syntax

Here, we'll discuss some important syntax for interacting with `nnsight` models. Since these models are extensions of HuggingFace models, some of this information (e.g. tokenization) applies to plain HuggingFace models as well as `nnsight` models, and some of it (e.g. forward passes) is specific to `nnsight`, i.e. it would work differently if you just had a standard HuggingFace model. Make sure to keep this distinction in mind, otherwise syntax can get confusing!

### Model config

Each model comes with a `model.config`, which contains lots of useful information about the model (e.g. number of heads and layers, size of hidden layers, etc.). You can access this with `model.config`. Run the code below to see this in action, and to define some useful variables for later.

In [ ]:
model = LanguageModel("EleutherAI/gpt-j-6b", device_map="auto", dtype=t.bfloat16)
tokenizer = model.tokenizer

N_HEADS = model.config.n_head
N_LAYERS = model.config.n_layer
D_MODEL = model.config.n_embd
D_HEAD = D_MODEL // N_HEADS

print(f"Number of heads: {N_HEADS}")
print(f"Number of layers: {N_LAYERS}")
print(f"Model dimension: {D_MODEL}")
print(f"Head dimension: {D_HEAD}\n")

print("Entire config: ", model.config)

### Tokenizers

A model comes with a tokenizer, accessable with `model.tokenizer` (just like TransformerLens). Unlike TransformerLens, we won't be using utility functions like `model.to_str_tokens`, instead we'll be using the tokenizer directly. Some important functions for today's exercises are:

* `tokenizer` (i.e. just calling it on some input)
    * This takes in a string (or list of strings) and returns the tokenized version.
    * It will return a dictionary, always containing `input_ids` (i.e. the actual tokens) but also other things which are specific to the transformer model (e.g. `attention_mask` - see dropdown).
    * Other useful arguments for this function:
        * `return_tensors` - if this is `"pt"`, you'll get results returned as PyTorch tensors, rather than lists (which is the default).
        * `padding` - if True (default is False), the tokenizer can accept sequences of variable length. The shorter sequences get padded at the beginning (see dropdown below for more).
* `tokenizer.decode`
    * This takes in tokens, and returns the decoded string.
    * If the input is an integer, it returns the corresponding string. If the input is a list / 1D array of integers, it returns all those strings concatenated (which can sometimes not be what you want).
* `tokenizer.batch_decode`
    * Equivalent to `tokenizer.decode`, but it doesn't concatenate.
    * If the input is a list / 1D integer array, it returns a list of strings. If the input is 2D, it will concatenate within each list.
* `tokenizer.tokenize`
    * Takes in a string, and returns a list of strings.

Run the code below to see some examples of these functions in action.

In [ ]:
# Calling tokenizer returns a dictionary, containing input ids & other data.
# If returned as a tensor, then by default it will have a batch dimension.
print(tokenizer("This must be Thursday", return_tensors="pt"))

# Decoding a list of integers, into a concatenated string.
print(tokenizer.decode([40, 1239, 714, 651, 262, 8181, 286, 48971, 12545, 13]))

# Using batch decode, on both 1D and 2D input.
print(tokenizer.batch_decode([4711, 2456, 481, 307, 6626, 510]))
print(tokenizer.batch_decode([[1212, 6827, 481, 307, 1978], [2396, 481, 428, 530]]))

# Split sentence into tokens (note we see the special Ġ character in place of prepended spaces).
print(tokenizer.tokenize("This sentence will be tokenized"))

<details>
<summary>Note on <code>attention_mask</code> (optional)</summary>

`attention_mask`, which is a series of 1s and 0s. We mask attention at all 0-positions (i.e. we don't allow these tokens to be attended to). This is useful when you have to do padding. For example:

```python
model.tokenizer(["Hello world", "Hello"], return_tensors="pt", padding=True)
```

will return:

```python
{
    'attention_mask': tensor([[1, 1], [0, 1]]),
    'input_ids': tensor([[15496,   995], [50256, 15496]])
}
```

We can see how the shorter sequence has been padded at the beginning, and attention to this token will be masked.

</details>

### Model outputs

At a high level, there are 2 ways to run our model: using the `trace` method (a single forward pass) and the `generate` method (generating multiple tokens). We'll focus on `trace` for now, and we'll discuss `generate` when it comes to multi-token generation later.

The default behaviour of forward passes in normal HuggingFace models is to return an object containing logits (and optionally a bunch of other things). The default behaviour of `trace` in `nnsight` is to not return anything, because anything that we choose to return is explicitly returned inside the context manager.

Below is the simplest example of code to run the model (and also access the internal states of the model). Run it and look at the output, then read the explanation below. Remember to obtain and set an API key first if you're using remote execution!

In [ ]:
# If you have an API key & want to work remotely, then set REMOTE = True and replace "YOUR-API-KEY"
# with your actual key. If not, then leave REMOTE = False.
REMOTE = False
if REMOTE:
    CONFIG.set_default_api_key("YOUR-API-KEY")

prompt = "The Eiffel Tower is in the city of"

with model.trace(prompt, remote=REMOTE):
    # Save the model's hidden states
    hidden_states = model.transformer.h[-1].output[0].save()

    # Save the model's logit output
    logits = model.lm_head.output[0, -1].save()

# Get the model's logit output, and it's next token prediction
print(f"logits.shape = {logits.shape} = (vocab_size,)")
print("Predicted token ID =", predicted_token_id := logits.argmax().item())
print(f"Predicted token = {tokenizer.decode(predicted_token_id)!r}")

# Print the shape of the model's residual stream
print(f"\nresid.shape = {hidden_states.shape} = (batch_size, seq_len, d_model)")

Lets go over this piece by piece.

**First, we create a context block** by calling `.trace(...)` on the model object. This denotes that we wish to generate tokens given some prompts.

```python
with model.trace(prompt, remote=REMOTE):
```

By default, running this will cause your model to be loaded & run locally, but by passing `remote=REMOTE`, it causes the model to be run on the server instead. This is very useful when working with models too large to fit on your machine (or even models which can fit on your machine, but run slowly due to their size, however if you're running this material on a sufficiently large GPU, you may prefer to set `REMOTE=False`).  The input argument can take a variety of formats: strings, lists of tokens, tensors of tokens, etc. Here, we've just used a string `prompt`.

The most interesting part of `nnsight` is the ability to access the model's internal states (like you might already have done with TransformerLens). Let's now see how this works!

```python
hidden_states = model.transformer.h[-1].output[0].save()
```

On this line we're saying: within our forward pass, access the last layer of the transformer `model.transformer.h[-1]`, access this layer's output `.output` (which is a tuple of tensors), index the first tensor in this tuple `.output[0]`, and save it `.save()`.

Let's break down this line in a bit more detail:

* `model.transformer.h[-1]` is a module in our transformer.
    * If you `print(model)`, you'll see that it consists of `transformer` and `lm_head` (for "language modelling head"). The `transformer` module is made up of embeddings & dropout, a series of layers (called `.h`, for "hidden states"), and a final layernorm. So indexing `.h[-1]` gives you the final layer.
    * Note - it's often useful to visit the documentation page for whatever model you're working on, e.g. you can find GPT-J [here](https://huggingface.co/transformers/v4.11.3/_modules/transformers/models/gptj/modeling_gptj.html). Not all models will have a nice uniform standardized architecture like you might be used to in TransformerLens!
* `.output[0]` gives you this module's output, as a **proxy**.
    * The output of a module is often a tuple (again, you can see on the [documentation page](https://huggingface.co/transformers/v4.11.3/_modules/transformers/models/gptj/modeling_gptj.html) what the output of each module is). In this case, it's a tuple of 2 tensors, the first of which is the actual layer output (the thing we want).
    * Doing operations on a proxy still returns a proxy - this is why we can index into the `output` proxy tuple and get a proxy tensor!
* `.save()` takes this proxy output, and returns the actual object (which you can now access outside the context manager).

<details>
<summary>A bit more detail on <code>save</code> (optional)</summary>

To be more specific, `.save()` informs the **intervention computational graph** to clone the value of a proxy, allowing us to access the value of a proxy after the forward pass.

During processing of the intervention computational graph we are building, when the value of a proxy is no longer needed, its value is dereferenced and destroyed. If you've saved it, then you'll be able to access the value of the proxy after this happens (i.e. outside the context manager).

</details>

<details>
<summary>Optional exercise - we mentioned that <code>.output</code> returns a tuple of 2 tensors. Can you use the <a href="https://huggingface.co/transformers/v4.11.3/_modules/transformers/models/gptj/modeling_gptj.html">documentation page</a> what the second tensor in this tuple is?</summary>

The second output is also a tuple of tensors, of length 2. In the GPT-J source code, they are called `present`. They represent the keys and values which were calculated in this forward pass (as opposed to those that were calculated in an earlier forward pass, and cached by the model). Since we're only generating one new token, these are just the full keys and values.

</details>

<br>

The next command:

```python
logits = model.lm_head.output[0, -1].save()
```

can be understood in a very similar way. The only difference is that we're accessing the output of `lm_head`, the language modelling head (i.e. the unembedding at the very end), and the output is just a tensor of shape `(batch, seq, d_vocab)` rather than a tuple of tensors. Again, see the [documentation page](https://huggingface.co/transformers/v4.11.3/_modules/transformers/models/gptj/modeling_gptj.html) for this.

If you've worked with Hugging Face models then you might be used to getting logits directly from the model output, but here we generally extract logits from the model internals just like any other activation because this allows us to **control exactly what we return.** If we return lots of very large tensors, this can take quite a while to download from the server (remember that `d_vocab` is often very large for transformers, i.e. around 50k). See the "which objects to save" section below for more discussion on this.

### Output vs input

You can also extract a module's input using `.input` or `.inputs`. If a module's forward method is called as `module.forward(*args, **kwargs)` then `.inputs` returns a tuple of `(tuple_of_args, dict_of_kwargs)`. Alternatively, `.input` is an alias for `.inputs[0][0]`, in other words it returns the first arg from the module's forward method (which is usually the tensor we want).

Remember that if you're not sure then you can debug with `print(module.input.shape)` - even if `.inputs` is a tuple of inputs, this will work to recursively print the shape of all the tensors in the tuple, rather than causing an error.

### Which objects to save

Note that we saved `logits` above, which is a vector of length 50k. In general, it's best to save as small an object as possible, because this reduces the size of object you'll have to download from the server. For example, if you only want the next token completions, just argmax the logits and then save the result! All basic tensor operations can be performed within your context manager.

## Putting this into practice

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "1.3.2.9"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part32_function_vectors_and_model_steering.solutions import generate_antonym_dataset, calculate_h, intervene_with_h, calculate_h_and_intervene, calculate_h_and_intervene_logprobs, calculate_fn_vectors_and_intervene, calculate_fn_vector, intervene_with_fn_vector


### Exercise - generalize results to another task (optional)

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 15-30 minutes on this exercise.
> ```

In this exercise, you get to pick a task different to the antonyms task, and see if the results still hold up (for the same set of attention heads).

We'll leave this exercise fairly open-ended, without any code templates for you to fill in. However, if you'd like some guidance you can use the dropdown below.

<details>
<summary>Guidance for exercise</summary>

Whatever your task, you'll want to generate a new set of words. You can repurpose the `generate_dataset` function from the antonyms task, by supplying a different prompt and initial set of examples (this will require generating & using an OpenAI api key, if you haven't already), or you can just find an appropriate dataset online.

When you define the `ICLDataset`, you might want to use `bidirectional=False`, if your task isn't symmetric. The antonym task is symmetric, but others (e.g. the Country-Capitals task) are not.

You'll need to supply a new prompt template for the `intervene_with_fn_vector` function, but otherwise most of your code should stay the same.

</details>

In [ ]:
with open(section_dir / "data/country_capital_pairs.txt", "r", encoding="utf-8") as f:
    COUNTRY_CAPITAL_PAIRS = [line.split() for line in f.readlines()]

country = "Netherlands"
_COUNTRY_CAPITAL_PAIRS = [pair for pair in COUNTRY_CAPITAL_PAIRS if pair[0] != country]

dataset = ICLDataset(_COUNTRY_CAPITAL_PAIRS, size=20, n_prepended=5, bidirectional=False)
head_list = [
    (8, 0),
    (8, 1),
    (9, 14),
    (11, 0),
    (12, 10),
    (13, 12),
    (13, 13),
    (14, 9),
    (15, 5),
    (16, 14),
]

fn_vector = calculate_fn_vector(model, dataset, head_list)

# Intervene with the function vector
completion, completion_intervention = intervene_with_fn_vector(
    model=model,
    word=country,
    layer=9,
    fn_vector=fn_vector,
    prompt_template="When you think of {x},",
    n_tokens=40,
)

table = Table("No intervention", "intervention")
table.add_row(repr(completion), repr(completion_intervention))
rprint(table)

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# This exercise has no automatic test — call `_dd_report_complete()`
# in a new cell once you're satisfied with your answer.
